# Mask bench: the checker in the decoder or after it

Qwen2.5-Instruct answers the 125 mask tasks, one prompt per task, under each decoding arm
(`masks/bench/decode.py` in the bundle says what each arm does). Every first-try failure gets one lint repair.
The notebook writes one jsonl per model and arm. Scoring is redone on the laptop by `report.py`.

**Input:** `bundle.tar.gz`, made on the laptop by `bash masks/bench/colab/bundle.sh` (code, views, tasks, both
grammars, the four built systems).
- **Colab:** put it in Drive as `MyDrive/mask_bench/bundle.tar.gz`. Results go to `MyDrive/mask_bench/results/`,
  so they survive a disconnect.
- **Kaggle:** add it as a dataset (any name), turn on GPU and Internet. Results go to `/kaggle/working/results/`.
  *Save Version → Save & Run All* runs it without the browser.

**Order:** Runtime → GPU (a T4 is enough). Run all with `SMOKE = True` (2 tasks per system, every arm) and
read the table at the end. Then set `SMOKE = False` and run all again. Rows are appended as they finish,
and a rerun skips rows already written, so a lost session resumes where it stopped.

**Back on the laptop:** copy the `*.jsonl` files (not `smoke/`) into `masks/bench/results/`.

In [ ]:
# Settings
MODELS = ["Qwen/Qwen2.5-7B-Instruct"]  # then "Qwen/Qwen2.5-14B-Instruct" if it fits (nf4: about 11 GB on the GPU)
QUANT = "nf4"  # 4-bit bitsandbytes. "fp16" needs 15 GB of weights for the 7B: Kaggle's two T4s, or a larger GPU
ARMS = [("free", "task"), ("grammar2", "task"), ("checked", "task"), ("grammar2", "sheet"), ("grammar1", "task")]
RUNS = 3
GPUS_PER_WORKER = 1  # 2 for a model one T4 cannot hold (Qwen3.8-27B in nf4 is about 20 GB)
SMOKE = True  # 2 tasks per system and 1 run of every arm, into results/smoke/
HOURS = 11  # then stop the workers and save what is written: Kaggle ends a session at 12 h

import os, time
T0 = time.time()
os.environ["HF_HUB_DISABLE_XET"] = "1"  # a xet download once stalled for 10 min (DEVLOG)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # less fragmentation; the workers inherit it

In [ ]:
# The bundle, unpacked to /tmp/maskbench; where results go
import glob, pathlib, subprocess, sys
KAGGLE = os.path.exists("/kaggle/working")
if KAGGLE:  # a dataset upload may arrive as the archive or already unpacked
    found = glob.glob("/kaggle/input/**/bundle.tar.gz", recursive=True)
    found += [str(pathlib.Path(p).parents[2]) for p in glob.glob("/kaggle/input/**/masks/bench/decode.py", recursive=True)]
    RESULTS = pathlib.Path("/kaggle/working/results")
    RESULTS.mkdir(parents=True, exist_ok=True)
    for f in glob.glob("/kaggle/input/**/results/*.jsonl", recursive=True):  # an earlier session's output, attached
        subprocess.run(["cp", "-n", f, str(RESULTS)], check=True)  # to resume
else:
    from google.colab import drive
    drive.mount("/content/drive")
    home = pathlib.Path("/content/drive/MyDrive/mask_bench")
    found = [str(p) for p in [home / "bundle.tar.gz"] if p.exists()]
    RESULTS = home / "results"
assert found, "bundle.tar.gz not found: see the first cell"
WORK = pathlib.Path("/tmp/maskbench")
WORK.mkdir(exist_ok=True)
if found[0].endswith(".tar.gz"):
    subprocess.run(["tar", "-xzf", found[0], "-C", str(WORK)], check=True)
else:
    subprocess.run(["cp", "-r", f"{found[0]}/.", str(WORK)], check=True)
BENCH = WORK / "masks" / "bench"
print(found[0])
print((BENCH / "colab" / "MANIFEST").read_text().splitlines()[0])

In [ ]:
# Python packages, pinned to the laptop's versions (bitsandbytes is Linux-only there)
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!df -h /root /tmp | tail -n +2  # the model download lands under /root/.cache
!pip install -q genlm-control==0.4.0 genlm-backend==0.3.0 genlm-bytes==0.2.0 genlm-grammar==0.2.0 \
    transformers==5.17.0 accelerate==1.15.0 regex==2026.9.10 bitsandbytes
!pip list 2>/dev/null | grep -E "^(torch|transformers|bitsandbytes|genlm-control|genlm-backend) " 

In [ ]:
%%bash
# AmberTools 24.8 for ambmask, the scorer and the checker potential: the laptop's build, in its own prefix
# (micromamba, so Colab's Python is untouched and no restart is needed). About 3 minutes.
set -e
if [ ! -x /tmp/amber/bin/ambmask ]; then
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /tmp bin/micromamba
  /tmp/bin/micromamba create -y -q -p /tmp/amber -c conda-forge "ambertools=24.8=cuda_None_nompi_py311*_101" \
    > /tmp/amber.log 2>&1 || { tail -20 /tmp/amber.log; exit 1; }
fi
ls /tmp/amber/conda-meta | grep ^ambertools

In [ ]:
# Before decoding: the systems are the checksummed ones and every gold reproduces with this ambmask
import json
from concurrent.futures import ThreadPoolExecutor
os.environ["AMBERHOME"] = "/tmp/amber"
sys.path.insert(0, str(BENCH))
import tasks

sums = dict(reversed(line.split()) for line in (BENCH / "SHA256SUMS").read_text().splitlines())
assert all(tasks.digest(s) == h for s, h in sums.items()), "a system differs from the one the gold was computed on"
rows = [json.loads(line) for line in (BENCH / "tasks.jsonl").read_text().splitlines()]
with ThreadPoolExecutor(8) as ex:
    got = list(ex.map(lambda r: tasks.select(r["system"], r["reference"])[0], rows))
bad = [r["id"] for r, a in zip(rows, got) if a is None or set(a) != tasks.from_ranges(r["gold"])]
assert not bad, bad
print(f"{len(rows)}/{len(rows)} golds reproduce with this ambmask, on systems matching SHA256SUMS")

In [ ]:
# The arms: one worker process per GPUS_PER_WORKER GPUs, each with its own copy of the model, the arms
# dealt out in turn (two T4s, one GPU each: free, checked, grammar1 on one; grammar2 and the sheet on the
# other). Rows are appended as they finish; rerunning this cell skips rows already written.
import time
from huggingface_hub import snapshot_download

OUT = RESULTS / "smoke" if SMOKE else RESULTS
OUT.mkdir(parents=True, exist_ok=True)
n_gpu = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines())
groups = [",".join(map(str, range(i, i + GPUS_PER_WORKER))) for i in range(0, n_gpu - GPUS_PER_WORKER + 1, GPUS_PER_WORKER)]
for name in MODELS:
    snapshot_download(name)  # once, before the workers start
    procs = []
    for w, gpus in enumerate(groups):
        arms = [f"{d}:{l}" for d, l in ARMS[w::len(groups)]]
        if arms:
            log = open(OUT / f"{name.split('/')[-1].lower()}-{QUANT}_gpu{gpus.replace(',', '-')}.log", "a")
            procs.append((log.name, subprocess.Popen(
                [sys.executable, str(BENCH / "decode.py"), name, QUANT, str(1 if SMOKE else RUNS), str(OUT),
                 "2" if SMOKE else "all", *arms],
                stdout=log, stderr=subprocess.STDOUT, env={**os.environ, "CUDA_VISIBLE_DEVICES": gpus})))
    while any(p.poll() is None for _, p in procs):
        time.sleep(60 if SMOKE else 600)
        if time.time() - T0 > HOURS * 3600:
            print("time budget spent: stopping the workers; rerun to resume", flush=True)
            for _, p in procs:
                p.terminate()
        for log, p in procs:
            lines = pathlib.Path(log).read_text().splitlines()
            print(time.strftime("%H:%M"), pathlib.Path(log).name, lines[-1][:160] if lines else "", flush=True)
    for log, p in procs:
        print(pathlib.Path(log).name, "exit", p.returncode)
        if p.returncode:
            print("\n".join(pathlib.Path(log).read_text().splitlines()[-30:]))

In [ ]:
# What was written (verdicts here are this box's; report.py on the laptop is the record)
import statistics
for f in sorted(OUT.glob("*.jsonl")):
    rs = [json.loads(line) for line in f.read_text().splitlines()]
    groups = {}  # seconds per prompt group, in the order written (systems outermost)
    for r in rs:
        groups.setdefault((r["system"], r["run"], r["system"] if r["layout"] == "sheet" else r["id"]), r["seconds"])
    keys = list(groups)
    first = [groups[k] for i, k in enumerate(keys) if i == 0 or keys[i - 1][0] != k[0]]  # these also prefilled a view
    later = [groups[k] for i, k in enumerate(keys) if i > 0 and keys[i - 1][0] == k[0]]
    print(f"{f.name:52s} {len(rs):4d} rows, first try {sum(r['verdict'] == 'EXACT' for r in rs):3d}, "
          f"after lint {sum(r['lint_verdict'] == 'EXACT' for r in rs):3d}, dead {sum(r['dead'] for r in rs):3d} "
          f"(out of tokens {sum(r['truncated'] for r in rs)}), errors {sum(bool(r['error']) for r in rs)}, "
          f"{sum(groups.values()) / 60:.1f} min; first group per system (with the view's prefill) "
          f"{[round(v) for v in first]} s, later groups median {statistics.median(later) if later else float('nan'):.1f} s")

In [ ]:
# One archive of the results, to download (on Colab they are also in Drive)
import shutil
archive = shutil.make_archive(str(RESULTS.parent / ("results_smoke" if SMOKE else "results")), "gztar", OUT)
print(archive)
if not KAGGLE:
    from google.colab import files
    files.download(archive)